In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import FloatType
from snowflake.snowpark import Window
import snowflake.snowpark.functions as F

session = get_active_session()

In [ ]:
df_transfer = session.table("BIGDATA_DB.RAW.TRANSFERS")
df_transfer.show(10)
df_transfer.print_schema()
df_transfer.count()

In [ ]:
df_transfer_sale = df_transfer.with_column(
    "value_float", 
    F.try_cast(F.col("value"), FloatType())
)


In [ ]:
df_token = session.table("BIGDATA_DB.STAGING.DIM_TOKENS_INDEXED")

df_join = df_transfer.join(df_token, df_token['ADDRESS'] == df_transfer['TOKEN_ADDRESS'], 'INNER')

df_join.show(10)

In [ ]:
df_final = df_join.with_column(
    "adjust_value",
    F.col('VALUE') / F.pow(10, F.col('DECIMALS'))
).select(
    "FROM_ADDRESS",
    'TO_ADDRESS',
    "TOKEN_ADDRESS",
    "NAME",
    "SYMBOL",
    "ADJUST_VALUE",
    'BLOCK_TIMESTAMP'
)
df_final.show(10)




In [ ]:
df_from = df_final.select(F.col("from_address").alias("user_address"))
df_to = df_final.select(F.col("to_address").alias("user_address"))

df_user = df_from.union_all(df_to).distinct()

df_map_user = df_user.with_column("user_id", F.row_number().over(Window.order_by('user_address')))
df_map_user.write.mode("overwrite").save_as_table(
    "BIGDATA_DB.STAGING.MAP_USER"
)
df_map_user.show()

In [ ]:
import snowflake.snowpark.functions as F

df_map_token = session.table("BIGDATA_DB.STAGING.MAP_TOKEN")
df_map_user = session.table("BIGDATA_DB.STAGING.MAP_USER")
map_from = df_map_user.alias("map_from")
map_to = df_map_user.alias("map_to")
map_token = df_map_token.alias("map_token")

df_indexed = (
    df_final
    .join(map_from, df_final["FROM_ADDRESS"] == map_from["USER_ADDRESS"], "inner")
    .join(map_to, df_final["TO_ADDRESS"] == map_to["USER_ADDRESS"], "inner")
    .join(map_token, df_final["TOKEN_ADDRESS"] == map_token["TOKEN_ADDRESS"], "inner")
)

df_final_matrix = df_indexed.select(
    map_from["USER_ID"].alias("FROM_USER_ID"),
    map_to["USER_ID"].alias("TO_USER_ID"),
    map_token["TOKEN_ID"].alias("TOKEN_ID"),
    F.col("ADJUST_VALUE"),
    F.col("BLOCK_TIMESTAMP")

)

df_final_matrix.write.mode("overwrite").save_as_table(
    "BIGDATA_DB.STAGING.TRANSFERS_INDEXED",
    table_type="transient"
)

In [ ]:
df_final_matrix.show()